# ⚙ **Basic Settings**

In [ ]:
# @title  {"display-mode":"form"}
# @markdown ▪️ 실습을 위해 필요한 **기본 라이브러리**들을 불러옵니다.
# @markdown
# @markdown ▪️ **왼쪽의 ▶ 버튼을 클릭**하여 셀을 실행시켜주세요. ( 시간이 약간 소요될 수 있습니다. )
# @markdown
# @markdown ▪️ "✅ 기본 설정이 완료되었습니다." 문구가 나올 때 까지 기다려주세요.

import time
import threading
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings("ignore")

# Datasets
from sklearn.datasets import load_iris, load_wine, load_breast_cancer, load_digits
from sklearn.datasets import load_diabetes, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.utils import Bunch
import pickle, base64
!pip install medmnist
clear_output()
import medmnist
from medmnist import INFO, OrganAMNIST

data = OrganAMNIST(split="val", download=True)
del data

# Models
from sklearn.linear_model import SGDRegressor, SGDClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Error Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error,root_mean_squared_error, r2_score
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

from matplotlib.colors import LinearSegmentedColormap

# 흰색에서 teal까지 그라디언트
colors = ['white', 'teal']

# LinearSegmentedColormap을 사용하여 사용자 정의 컬러맵 생성
custom_cmap = LinearSegmentedColormap.from_list('white_to_teal', colors)

def load_organamnist(as_frame=False, limit=442):
    info = INFO["organamnist"]
    DataClass = getattr(medmnist, info["python_class"])

    ds = DataClass(split="val", download=True)

    # imgs: (N, 28, 28, 1) → (N, 28, 28)
    X = ds.imgs
    if X.ndim == 4 and X.shape[-1] == 1:
        X = X.squeeze(-1)

    # flatten (N, 784)
    X_flat = X.reshape(len(X), -1)
    y = ds.labels.squeeze()

    # limit 적용
    if limit is not None and limit < len(X_flat):
        X = X[:limit]
        X_flat = X_flat[:limit]
        y = y[:limit]

    # feature names 2차원식 생성
    h, w = X.shape[1], X.shape[2]   # 보통 28, 28
    feature_names = [f"pixel_{i}_{j}" for i in range(h) for j in range(w)]

    # target_names (meaning of labels)
    label_map = info.get("label", {})
    if isinstance(label_map, dict) and label_map:
        keys_sorted = sorted(label_map.keys(), key=lambda k: int(k))
        target_names = [label_map[k] for k in keys_sorted]
    else:
        target_names = [str(i) for i in range(len(np.unique(y)))]

    return Bunch(
        data=X_flat,            # (N, 784)
        target=y,
        images=X,               # (N, 28, 28)
        target_names=target_names,
        feature_names=feature_names,  # "pixel_row_col"
        DESCR=f"""
        The OrganAMNIST is based on 3D computed tomography (CT) images from Liver Tumor Segmentation Benchmark (LiTS).
        It is renamed from OrganMNIST_Axial (in MedMNIST v1) for simplicity.
        We use bounding-box annotations of 11 body organs from another study to obtain the organ labels.
        Hounsfield-Unit (HU) of the 3D images are transformed into gray-scale with an abdominal window.
        We crop 2D images from the center slices of the 3D bounding boxes in axial views (planes).
        The images are resized into 1×28×28 to perform multi-class classification of 11 body organs.
        115 and 16 CT scans from the source training set are used as training and validation set, respectively.
        The 70 CT scans from the source test set are treated as the test set.
        Visit "https://arxiv.org/abs/2110.14795" for more information.
        """
    )

def print_info(td = 0.3, result = False):
    global task
    if not result:
        try:
            print(f'  ◻️ 데이터셋 :')
        except:
            print('\n  ⛔ 데이터셋을 선택해주세요.')
            return False

        time.sleep(td)
        try:
            clear_output(wait = True)
            print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
            print(f"  ◻️ 선택한 특성 : ")
        except:
            print('\n  ⛔ 학습에 활용할 특성을 선택해주세요.')
            return False

        time.sleep(td)
        try:
            clear_output(wait = True)
            print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
            print(f"  ✅ 선택한 특성 (총 {len(selected_columns)} 개) : \n     {selected_columns}")
            print(f"  ◻️ 데이터 분할 비율 :")
            print(f"     ├─ Train :  ")
            print(f"     ├─ Valid :  ")
            print(f"     └─ Test  : ")
            print(f"  ◻️ 스케일링 방법 : ")
        except:
            print('\n  ⛔ 데이터 분할 비율과 스케일링 방법을 선택해주세요.')
            return False

        time.sleep(td)
        try:
            clear_output(wait = True)
            print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
            print(f"  ✅ 선택한 특성 (총 {len(selected_columns)} 개) : \n     {selected_columns}")
            print(f"  ✅ 데이터 분할 비율 :")
            print(f"     ├─ Train :  {train_ratio:.2f}   ( {len(y_train)} 행 )")
            print(f"     ├─ Valid :  {valid_ratio:.2f}   ( {len(y_valid)} 행 )")
            print(f"     └─ Test  :  {test_ratio:.2f}   ( {len(y_test)} 행 )")
            print(f"  ✅ 스케일링 방법 : {scaler_print}({스케일링_방법})")
            print(f"  ◻️ 문제 유형 : ")
            print(f"  ◻️ 모델 종류 : ")
        except:
            print('\n  ⛔ 적절한 모델을 선택해주세요.')
            return False

        time.sleep(td)
        try:
            clear_output(wait = True)
            print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
            print(f"  ✅ 선택한 특성 (총 {len(selected_columns)} 개) : \n     {selected_columns}")
            print(f"  ✅ 데이터 분할 비율 :")
            print(f"     ├─ Train :  {train_ratio:.2f}   ( {len(y_train)} 행 )")
            print(f"     ├─ Valid :  {valid_ratio:.2f}   ( {len(y_valid)} 행 )")
            print(f"     └─ Test  :  {test_ratio:.2f}   ( {len(y_test)} 행 )")
            print(f"  ✅ 스케일링 방법 : {scaler_print}({스케일링_방법})")
            print(f"  ✅ 문제 유형 : {task}")
            print(f"  ◻️ 모델 종류 : {model_info['model_type']}")
            if model_info['degree'] >= 2:
                print(f"  ◻️ 모델 차수 : {model_info['degree']}")
            if model_info['model_type'] == '랜덤 포레스트':
                print(f"  ◻️ Criterion : {model_info['criterion']}")
                print(f"  ◻️ Max depth : {model_info['max_depth']}")
                print(f"  ◻️ N. of Trees : {model_info['n_estimators']}")
            else:
                print(f"  ◻️ 손실 함수 : {model_info['metric']}")
                print(f"  ◻️ 에포크 수 : ")
                print(f"  ◻️ 학습률 : ")
            if task != model_info['task'] : raise Exception('error')
        except:
            print('\n  ⛔ 문제 유형에 맞는 모델을 선택해주세요.')
            return False

        time.sleep(td)

        clear_output(wait = True)
        print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
        print(f"  ✅ 선택한 특성 (총 {len(selected_columns)} 개) : \n     {selected_columns}")
        print(f"  ✅ 데이터 분할 비율 :")
        print(f"     ├─ Train :  {train_ratio:.2f}   ( {len(y_train)} 행 )")
        print(f"     ├─ Valid :  {valid_ratio:.2f}   ( {len(y_valid)} 행 )")
        print(f"     └─ Test  :  {test_ratio:.2f}   ( {len(y_test)} 행 )")
        print(f"  ✅ 스케일링 방법 : {scaler_print}({스케일링_방법})")
        print(f"  ✅ 문제 유형 : {task}")
        print(f"  ✅ 모델 종류 : {model_info['model_type']}")
        if model_info['degree'] >= 2:
            print(f"  ✅ 모델 차수 : {model_info['degree']}")
        if model_info['model_type'] == '랜덤 포레스트':
            print(f"  ✅ Criterion : {model_info['criterion']}")
            print(f"  ✅ Max depth : {model_info['max_depth']}")
            print(f"  ✅ N. of Trees : {model_info['n_estimators']}")
        else:
            print(f"  ✅ 손실 함수 : {model_info['metric']}")
            print(f"  ◻️ 에포크 수 : ")
            print(f"  ◻️ 학습률 : ")


        time.sleep(td)
        clear_output(wait = True)
        print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
        print(f"  ✅ 선택한 특성 (총 {len(selected_columns)} 개) : \n     {selected_columns}")
        print(f"  ✅ 데이터 분할 비율 :")
        print(f"     ├─ Train :  {train_ratio:.2f}   ( {len(y_train)} 행 )")
        print(f"     ├─ Valid :  {valid_ratio:.2f}   ( {len(y_valid)} 행 )")
        print(f"     └─ Test  :  {test_ratio:.2f}   ( {len(y_test)} 행 )")
        print(f"  ✅ 스케일링 방법 : {scaler_print}({스케일링_방법})")
        print(f"  ✅ 문제 유형 : {model_info['task']}")
        print(f"  ✅ 모델 종류 : {model_info['model_type']}")
        if model_info['degree'] >= 2:
            print(f"  ✅ 모델 차수 : {model_info['degree']}")
        if model_info['model_type'] == '랜덤 포레스트':
            print(f"  ✅ criterion : {model_info['criterion']}")
            print(f"  ✅ Max depth : {model_info['max_depth']}")
            print(f"  ✅ N. of Trees : {model_info['n_estimators']}")
        else:
            print(f"  ✅ 손실 함수 : {model_info['metric']}")
            print(f"  ✅ 에포크 수 : {model_info['epochs']}")
            print(f"  ✅ 학습률 : {model_info['learning_rate']}")

        time.sleep(td)
        print('\n  👏 학습 준비가 완료되었습니다.')
        return True
    else:
        print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')
        print(f"  ✅ 선택한 특성 (총 {len(selected_columns)} 개) : \n       {selected_columns}")
        print(f"  ✅ 데이터 분할 비율 :")
        print(f"       ├─ Train :  {train_ratio:.2f}   ( {len(y_train)} 행 )")
        print(f"       ├─ Valid :  {valid_ratio:.2f}   ( {len(y_valid)} 행 )")
        print(f"       └─ Test  :  {test_ratio:.2f}   ( {len(y_test)} 행 )")
        print(f"  ✅ 스케일링 방법 : {scaler_print}({스케일링_방법})")
        print(f"  ✅ 문제 유형 : {model_info['task']}")
        print(f"  ✅ 모델 종류 : {model_info['model_type']}")
        if model_info['degree'] >= 2:
            print(f"  ✅ 모델 차수 : {model_info['degree']}")
        if model_info['model_type'] == '랜덤 포레스트':
            print(f"  ✅ criterion : {model_info['criterion']}")
            print(f"  ✅ Max depth : {model_info['max_depth']}")
            print(f"  ✅ N. of Trees : {model_info['n_estimators']}")
        else:
            print(f"  ✅ 손실 함수 : {model_info['metric']}")
            print(f"  ✅ 에포크 수 : {model_info['epochs']}")
            print(f"  ✅ 학습률 : {model_info['learning_rate']}")
        print()

clear_output()
print('  ✅ 기본 설정이 완료되었습니다.')

---
# **📊 데이터, 문제 선택**

In [ ]:
# @markdown ▪️ 실습에 활용할 **데이터셋을 선택**합니다.
# @markdown
# @markdown ▪️ **원하는 데이터셋을 선택**한 후 **왼쪽의 ▶ 버튼**을 클릭하여 셀을 실행시켜주세요.
# @markdown
# @markdown ▪️ **데이터셋 설명**을 확인하고 싶다면 **[description]을 체크**하고 셀을 실행시켜주세요.

def load_dataset(dataset, description):

    if dataset == 'iris':
        loaded_data = load_iris()
    elif dataset == 'wine':
        loaded_data = load_wine()
    elif dataset == 'breast_cancer':
        loaded_data = load_breast_cancer()
    elif dataset == 'digits':
        loaded_data = load_digits()
    elif dataset == 'diabetes':
        loaded_data = load_diabetes()
    elif dataset == 'california_housing':
        loaded_data = fetch_california_housing()
    elif dataset == 'organ_mnist':
        loaded_data = load_organamnist()

    try:
        X = pd.DataFrame(loaded_data['data'], columns = loaded_data['feature_names'])
        Y = pd.DataFrame(loaded_data['target'], columns = ['target'])
        df = pd.concat([X, Y], axis = 1)
        if description: print(loaded_data['DESCR'])

    except:
        X = pd.DataFrame(loaded_data['data'], columns = loaded_data['feature_names'])
        Y = pd.DataFrame(loaded_data['target'], columns = ['target'])
        df = pd.concat([X, Y], axis = 1)
        if description: print(loaded_data['DESCR'])

    return loaded_data, X, Y, df

dataset = "organ_mnist"  #@param ["digits", "diabetes", "iris", "wine", "breast_cancer", "california_housing", "organ_mnist"]
description = True #@param {type:"boolean"}

data, X, Y, df = load_dataset(dataset, description)

print(f'  ✅ 데이터셋 : {dataset} data ( 총 {len(df)} 행 )')

df.head(5)

In [ ]:
# @markdown ▪️ 선택한 데이터셋에 맞는 **문제 유형**을 선택한 후 **왼쪽의 ▶ 버튼**을 클릭하여 셀을 실행시켜주세요.
task = "분류"  #@param ["분류", "회귀"]

print(f'  ✅ 문제 유형 : {task}')

---
# **🛠 모델 학습을 위한 데이터 전처리**

## **📌 특징 선택**

In [ ]:
# @markdown ▪️ 선택한 데이터셋에서 **학습에 활용할 특징 값들**을 선택합니다.
# @markdown
# @markdown ▪️ **왼쪽의 ▶ 버튼**을 클릭하여 셀을 실행시킨 후, **원하는 특징들을 체크**하고 **[선택완료] 버튼**을 눌러주세요.
# @markdown
# @markdown ❗**[시각화]를 체크**하면 선택된 **특징들의 간단한 분포를 시각화**합니다. 이 때, 선택된 특징이 많으면 시간이 다소 소요될 수 있습니다.

# --- 추가: 시각화 허용 최대 특성 수 설정 ---
MAX_VIZ_FEATURES = 100  # 100개 초과 시 시각화 금지

# 시각화 여부를 선택할 수 있는 체크박스 추가
visualize_checkbox = widgets.Checkbox(
    value=False,
    description='시각화',
    layout=widgets.Layout(width='200px')
)

def create_checkboxes(columns, all_checked=False):
    checkboxes = []
    for col in columns:
        checkbox = widgets.Checkbox(value=all_checked, description=col)
        checkboxes.append(checkbox)
    return checkboxes

# 체크박스 UI 생성
checkboxes = create_checkboxes(X.columns, all_checked=True)
select_button = widgets.Button(
    description="📌 선택 완료",
    button_style='success',
    layout=widgets.Layout(width='300px', margin='10px 0px 0px 0px')
)
select_button.style.button_color = '#00AA98'

selected_columns = []

# 로딩 애니메이션 표시 함수
def dynamic_loading_message(stop_event):
    animation1 = ['■□□□□ ⌛', '■■□□□ ⌛', '■■■□□ ⏳', '■■■■□ ⏳', '■■■■■ ⏳', '□■■■■ ⏳', '□□■■■ ⏳', '□□□■■ ⌛', '□□□□■ ⌛']
    animation2 = ['□□□□□', '□□□□□', '□□□□□', '□□□□□', '□□□□□', '■□□□□', '■■□□□', '■■■□□', '■■■■□']
    i = 0
    while not stop_event.is_set():
        clear_output(wait=True)
        print(f"  {animation1[i % len(animation1)]} 선택된 데이터를 시각화하고 있습니다. 잠시만 기다려주세요. {animation2[i % len(animation2)]}")
        print(f"\n  ✅ 선택한 특성 (총 {len(selected_columns)}개) : \n     {selected_columns}")
        i += 1
        time.sleep(0.5)

# 시각화 함수
def plot_data_distribution(data):
    stop_event = threading.Event()

    # 로딩 메시지 표시
    loading_thread = threading.Thread(target=dynamic_loading_message, args=(stop_event,))
    loading_thread.start()

    # 데이터 시각화
    if task == '분류':
        sns.pairplot(data, hue='Y', diag_kind="hist", palette="Set2", plot_kws={'alpha': 0.7})
    else:
        sns.pairplot(data, plot_kws={'color': 'royalblue', 'alpha': 0.7})

    # 로딩 메시지 중단
    stop_event.set()
    loading_thread.join()

    clear_output(wait=True)
    plt.show()

    print(f"\n  ✅ 선택한 특성 (총 {len(selected_columns)}개) : \n     {selected_columns}")

# 선택 완료 버튼을 눌렀을 때 실행되는 함수
def on_select_click(b):
    global selected_columns
    selected_columns = [cb.description for cb in checkboxes if cb.value]

    # 선택된 특성으로 새로운 X, Y 생성
    x = X[selected_columns].copy()
    y = Y.copy()
    y = y.to_numpy().ravel()

    # 기존 출력 제거하고 선택된 특성 출력
    clear_output(wait=True)
    print(f"  ✅ 선택한 특성 (총 {len(selected_columns)}개) : \n     {selected_columns}")

    # --- 추가: 시각화 조건 검사 ---
    if visualize_checkbox.value:
        if len(selected_columns) > MAX_VIZ_FEATURES:
            # 100개 초과 시 시각화 금지 및 안내
            print("\n  ⚠️ 메모리 초과로 시각화 할 수 없다")
        else:
            # 시각화 실행
            plot_data = x.copy()
            plot_data['Y'] = y
            plot_data_distribution(plot_data)

# 버튼 클릭 이벤트 등록
select_button.on_click(on_select_click)

# UI 레이아웃 설정 (시각화 체크박스를 포함)
ui = widgets.VBox([widgets.VBox(checkboxes), visualize_checkbox, select_button])
display(ui)


# # @markdown ▪️ 선택한 데이터셋에서 **학습에 활용할 특징 값들**을 선택합니다.
# # @markdown
# # @markdown ▪️ **왼쪽의 ▶ 버튼**을 클릭하여 셀을 실행시킨 후, **원하는 특징들을 체크**하고 **[선택완료] 버튼**을 눌러주세요.
# # @markdown
# # @markdown ❗**[시각화]를 체크**하면 선택된 **특징들의 간단한 분포를 시각화**합니다. 이 때, 선택된 특징이 많으면 시간이 다소 소요될 수 있습니다.

# # 시각화 여부를 선택할 수 있는 체크박스 추가
# visualize_checkbox = widgets.Checkbox(
#     value=False,
#     description='시각화',
#     layout=widgets.Layout(width='200px')
# )

# def create_checkboxes(columns, all_checked=False):
#     checkboxes = []
#     for col in columns:
#         checkbox = widgets.Checkbox(value=all_checked, description=col)
#         checkboxes.append(checkbox)
#     return checkboxes

# # 체크박스 UI 생성
# checkboxes = create_checkboxes(X.columns, all_checked=True)
# select_button = widgets.Button(
#     description="📌 선택 완료",
#     button_style = 'success',
#     layout=widgets.Layout(width='300px', margin='10px 0px 0px 0px')
#     )

# select_button.style.button_color = '#00AA98'

# selected_columns = []

# # 로딩 애니메이션 표시 함수
# def dynamic_loading_message(stop_event):
#     animation1 = ['■□□□□ ⌛', '■■□□□ ⌛', '■■■□□ ⏳', '■■■■□ ⏳', '■■■■■ ⏳', '□■■■■ ⏳', '□□■■■ ⏳', '□□□■■ ⌛', '□□□□■ ⌛']
#     animation2 = ['□□□□□', '□□□□□', '□□□□□', '□□□□□', '□□□□□', '■□□□□', '■■□□□', '■■■□□', '■■■■□']
#     i = 0
#     while not stop_event.is_set():
#         clear_output(wait=True)
#         print(f"  {animation1[i % len(animation1)]} 선택된 데이터를 시각화하고 있습니다. 잠시만 기다려주세요. {animation2[i % len(animation2)]}")
#         print(f"\n  ✅ 선택한 특성 (총 {len(selected_columns)}개) : \n     {selected_columns}")
#         i += 1
#         time.sleep(0.5)

# # 시각화 함수
# def plot_data_distribution(data):
#     stop_event = threading.Event()

#     # 로딩 메시지 표시
#     loading_thread = threading.Thread(target=dynamic_loading_message, args=(stop_event,))
#     loading_thread.start()

#     # 데이터 시각화
#     if task == '분류':
#         sns.pairplot(data, hue='Y', diag_kind="hist", palette="Set2", plot_kws={'alpha': 0.7})
#     else:
#         sns.pairplot(data, plot_kws={'color': 'royalblue', 'alpha': 0.7})

#     # 로딩 메시지 중단
#     stop_event.set()
#     loading_thread.join()

#     clear_output(wait=True)
#     plt.show()

#     print(f"\n  ✅ 선택한 특성 (총 {len(selected_columns)}개) : \n     {selected_columns}")

# # 선택 완료 버튼을 눌렀을 때 실행되는 함수
# def on_select_click(b):
#     global selected_columns
#     selected_columns = [cb.description for cb in checkboxes if cb.value]

#     # 선택된 특성으로 새로운 X, Y 생성
#     x = X[selected_columns].copy()
#     y = Y.copy()
#     y = y.to_numpy().ravel()

#     # 기존 출력 제거하고 선택된 특성 출력
#     clear_output(wait=True)

#     print(f"  ✅ 선택한 특성 (총 {len(selected_columns)}개) : \n     {selected_columns}")

#     # 시각화 체크박스가 체크된 경우에만 시각화 실행
#     if visualize_checkbox.value:
#         plot_data = x.copy()
#         plot_data['Y'] = y
#         plot_data_distribution(plot_data)

# # 버튼 클릭 이벤트 등록
# select_button.on_click(on_select_click)

# # UI 레이아웃 설정 (시각화 체크박스를 포함)
# ui = widgets.VBox([widgets.VBox(checkboxes), visualize_checkbox, select_button])
# display(ui)


## **📏 데이터 분할 및 스케일링**

In [ ]:
# Drop the unselected columns
x = X[selected_columns].copy()
y = Y.copy()
y = y.to_numpy().ravel()

# @markdown ▪️ **Train/Valid/Test 비율**은 **6:2:2**로 고정되어 있습니다.

학습데이터_비율 = 0.6
train_ratio = 학습데이터_비율
# @markdown ▪️ **데이터 스케일링 방법**을 선택한 후, **왼쪽의 ▶ 버튼**을 클릭하여 셀을 실행시켜주세요.
스케일링_방법 = 'MinMaxScaler' #@param ["StandardScaler", "MinMaxScaler"]


valid_test_ratio = 1.0 - 학습데이터_비율
valid_ratio = round(valid_test_ratio / 2, 2)
test_ratio = round(valid_test_ratio / 2, 2)

# Train / Valid / Test Split
def split_data(x, y, train_size=0.6, valid_size=0.2, test_size=0.2):
    assert train_size + valid_size + test_size == 1.0

    x_train_valid, x_test, y_train_valid, y_test = train_test_split(x, y, test_size=test_size, random_state=42)

    valid_ratio = valid_size / (train_size + valid_size)
    x_train, x_valid, y_train, y_valid = train_test_split(x_train_valid, y_train_valid, test_size=valid_ratio, random_state=42)

    return x_train, x_valid, x_test, y_train, y_valid, y_test

x_train, x_valid, x_test, y_train, y_valid, y_test = split_data(x, y, train_size=train_ratio, valid_size=valid_ratio, test_size=test_ratio)


if 스케일링_방법 == 'StandardScaler':
    scaler = StandardScaler()
    scaler_print = '표준화'
elif 스케일링_방법 == 'MinMaxScaler':
    scaler = MinMaxScaler()
    scaler_print = '최대-최소 정규화'

print(f"  ✅ 데이터 분할 비율 :")
print(f"     ├─ Train :  {train_ratio:.2f}   ( {len(y_train)} 행 )")
print(f"     ├─ Valid :  {valid_ratio:.2f}   ( {len(y_valid)} 행 )")
print(f"     └─ Test  :  {test_ratio:.2f}   ( {len(y_test)} 행 )")
print(f"  ✅ 스케일링 방법 : {scaler_print}({스케일링_방법})")

---
# **💫 모델 선택**

In [ ]:
# @markdown ▪️ **왼쪽의 ▶ 버튼**을 클릭하여 셀을 실행시킨 후, 데이터에 맞는 적절한 **모델, 손실함수**를 선택하세요.
# @markdown
# @markdown ❗ ( 특징의 개수가 많고, 다항 회귀/분류 모델의 차수를 너무 높히면 세션이 종료되거나 모델 학습에 매우 많은 시간이 소요될 수 있습니다. )

# 전역 변수로 모델과 학습 파라미터를 저장
model_info = {}

# 공통 레이아웃 설정
widget_width = '300px'
description_width = '100px'

# 모델 선택 드롭다운
model_dropdown = widgets.Dropdown(
    options=[],
    value=None,
    description='모델 종류 :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)

# 손실 함수 드롭다운 (HBox로 감싸기)
metric_dropdown = widgets.Dropdown(
    options=[],
    value=None,
    description='손실 함수 :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)
metric_box = widgets.HBox([metric_dropdown])

# 다항 차수 슬라이더 (HBox로 감싸기)
degree_slider = widgets.IntSlider(
    value=2,
    min=2,
    max=9,
    step=1,
    description='다항 차수 :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)
degree_box = widgets.HBox([degree_slider])

# 손실 함수와 다항 차수 슬라이더를 하나의 VBox로 묶음
metric_degree_box = widgets.VBox([metric_box, degree_box])
metric_degree_box.layout.display = 'none'  # 기본적으로 숨김

# 학습률 입력 박스
lr_input = widgets.FloatText(
    value=0.5,
    step = 0.01,
    description='Learning Rate :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)

# 에포크 입력 박스
epoch_input = widgets.IntText(
    value=100,
    description='Epochs :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)

# 학습률과 에포크 입력 박스를 하나의 VBox로 묶음
training_params_box = widgets.VBox([epoch_input, lr_input])
training_params_box.layout.display = 'none'  # 기본적으로 숨김

# 랜덤 포레스트 파라미터 슬라이더
max_depth_slider = widgets.IntSlider(
    value=50,
    min=1,
    max=100,
    step=1,
    description='Max Depth :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)

tree_number_slider = widgets.IntSlider(
    value=2,
    min=1,
    max=500,
    step=1,
    description='N. of Trees:',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)

# Criterion 드롭다운
criterion_dropdown = widgets.Dropdown(
    options=[],
    value=None,
    description='Criterion :',
    layout=widgets.Layout(width=widget_width),
    style={'description_width': description_width}
)

# 랜덤 포레스트 옵션 박스
forest_box = widgets.VBox([criterion_dropdown, max_depth_slider, tree_number_slider])

# 모델 정보 출력
output = widgets.Output()

# 버튼: 모델 정보 및 학습 파라미터 저장
save_button = widgets.Button(
    description="📌 선택 완료",
    button_style='success',
    layout=widgets.Layout(width=widget_width, margin='10px 0px 0px 0px')
)
save_button.style.button_color = '#00AA98'

# task에 따라 모델과 메트릭 옵션을 설정하는 함수
def set_task_options(task):
    if task == '회귀':
        model_dropdown.options = ['선형회귀', '비선형회귀 (다항 회귀)', '랜덤 포레스트']
        metric_dropdown.options = ['mean_squared_error', 'root_mean_squared_error', 'mean_absolute_error']
    elif task == '분류':
        model_dropdown.options = ['선형분류 (로지스틱 회귀)', '비선형분류 (다항 로지스틱 회귀)', '랜덤 포레스트']
        metric_dropdown.options = ['log_loss (cross entropy)']

# 모델 선택에 따라 손실 함수와 다항 차수 슬라이더 및 랜덤 포레스트 옵션 표시/숨기기
def update_display(change):
    model_type = model_dropdown.value

    # 초기화: 모든 박스를 숨김
    metric_degree_box.layout.display = 'none'
    forest_box.layout.display = 'none'
    training_params_box.layout.display = 'none'

    if task == '분류':
        if model_type == '선형분류 (로지스틱 회귀)':
            metric_dropdown.options = ['log_loss (cross entropy)']
            metric_degree_box.layout.display = 'block'
            training_params_box.layout.display = 'block'
            degree_box.layout.display = 'none'
        elif model_type == '비선형분류 (다항 로지스틱 회귀)':
            metric_dropdown.options = ['log_loss (cross entropy)']
            metric_degree_box.layout.display = 'block'
            training_params_box.layout.display = 'block'
            degree_box.layout.display = 'block'
        elif model_type == '랜덤 포레스트':
            criterion_dropdown.options = ['gini', 'entropy']
            forest_box.layout.display = 'block'
    elif task == '회귀':
        if model_type == '선형회귀':
            metric_dropdown.options = ['mean_squared_error', 'root_mean_squared_error', 'mean_absolute_error']
            metric_degree_box.layout.display = 'block'
            training_params_box.layout.display = 'block'
            degree_box.layout.display = 'none'
        elif model_type == '비선형회귀 (다항 회귀)':
            metric_dropdown.options = ['mean_squared_error', 'root_mean_squared_error', 'mean_absolute_error']
            metric_degree_box.layout.display = 'block'
            training_params_box.layout.display = 'block'
            degree_box.layout.display = 'block'
        elif model_type == '랜덤 포레스트':
            criterion_dropdown.options = ['squared_error', 'absolute_error'] #['poisson', 'squared_error', 'friedman_mse', 'absolute_error']
            forest_box.layout.display = 'block'

# 최종 모델 및 학습 파라미터 선택 후 저장
def save_model_info(b):
    global model_info

    clear_output()  # 이전 출력 지우기
    model_type = model_dropdown.value
    degree = degree_slider.value
    metric = metric_dropdown.value
    lr = lr_input.value
    epochs = epoch_input.value

    # 모델 정보 및 학습 파라미터를 딕셔너리로 저장
    model_info = {
        'task': task,
        'model_type': model_type,
        'degree': degree if model_type in ['비선형회귀 (다항 회귀)', '비선형분류 (다항 로지스틱 회귀)'] else 1,
        'metric': metric if model_type != '랜덤 포레스트' else None,
        'learning_rate': lr if model_type != '랜덤 포레스트' else None,
        'epochs': epochs if model_type != '랜덤 포레스트' else None,
    }

    # 랜덤 포레스트 관련 파라미터 추가
    if model_type == '랜덤 포레스트':
        model_info['criterion'] = criterion_dropdown.value
        model_info['max_depth'] = max_depth_slider.value
        model_info['n_estimators'] = tree_number_slider.value

    # 출력 내용
    print(f"  ✅ 문제 유형 : {task}")
    print(f"  ✅ 모델 종류 : {model_type}")
    if model_type != '랜덤 포레스트':
        print(f"  ✅ 손실 함수 : {metric}")
        print(f"  ✅ 에포크 수 : {epochs}")
        print(f"  ✅ 학습률 : {lr}")
    if model_info['degree'] != 1:
        print(f"  ✅ 다항 차수 : {degree}")
    if model_type == '랜덤 포레스트':
        print(f"  ✅ Criterion : {model_info['criterion']}")
        print(f"  ✅ Max Depth : {model_info['max_depth']}")
        print(f"  ✅ N. of Trees : {model_info['n_estimators']}")

# 모델 타입 선택 시 표시할 박스 설정
model_dropdown.observe(update_display, names='value')

# Save 버튼 클릭 시 모델 및 학습 파라미터 저장
save_button.on_click(save_model_info)

# task에 따라 모델과 메트릭 옵션을 설정
set_task_options(task)

# 최종 UI 구성
display(widgets.VBox([model_dropdown, metric_degree_box, training_params_box, forest_box, save_button]), output)


---
# **🪄 모델 학습 및 결과 확인**

In [ ]:
# @title
# @markdown ▪️ **왼쪽의 ▶ 버튼**을 클릭하여 설정한 모델, 파라미터 등이 정확한지 확인하세요.
# @markdown
# @markdown ▪️ 확인 후 **Start Training** 버튼을 클릭하면 모델 학습이 시작됩니다.

def create_model(model_info):
    if model_info['task'] == '회귀':
        if model_info['model_type'] == '선형회귀':
            return SGDRegressor(max_iter=1, tol=None, learning_rate='constant', eta0=model_info['learning_rate'], random_state=42)
        elif model_info['model_type'] == '비선형회귀 (다항 회귀)':
            return SGDRegressor(max_iter=1, tol=None, learning_rate='constant', eta0=model_info['learning_rate'], random_state=42)
        elif model_info['model_type'] == '랜덤 포레스트':
            return RandomForestRegressor(criterion=model_info['criterion'], max_depth=model_info['max_depth'], n_estimators=model_info['n_estimators'], random_state=42)
    else:
        if model_info['model_type'] in ['선형분류 (로지스틱 회귀)', '비선형분류 (다항 로지스틱 회귀)']:
            return SGDClassifier(loss='log_loss', max_iter=1, tol=None, learning_rate='constant', eta0=model_info['learning_rate'], random_state=42)
        elif model_info['model_type'] in ['선형분류 (SVM)', '비선형분류 (다항 SVM)']:
            return SGDClassifier(loss='hinge', max_iter=1, tol=None, learning_rate='constant', eta0=model_info['learning_rate'], random_state=42)
        elif model_info['model_type'] == '랜덤 포레스트':
            return RandomForestClassifier(criterion=model_info['criterion'], max_depth=model_info['max_depth'], n_estimators=model_info['n_estimators'], random_state=42)


def preprocess_data(model_info, x_train, x_valid, x_test, scaler):
    if model_info['model_type'] in ['비선형회귀 (다항 회귀)', '비선형분류 (다항 로지스틱 회귀)', '비선형분류 (다항 SVM)']:
        poly_features = PolynomialFeatures(degree=model_info['degree'], include_bias=False)
        x_train_poly = poly_features.fit_transform(x_train)
        x_valid_poly = poly_features.transform(x_valid)
        x_test_poly = poly_features.transform(x_test)
    else:
        x_train_poly = x_train
        x_valid_poly = x_valid
        x_test_poly = x_test

    # Data Scaling
    x_train_scaled = scaler.fit_transform(x_train_poly)
    x_valid_scaled = scaler.transform(x_valid_poly)
    x_test_scaled = scaler.transform(x_test_poly)

    return x_train_scaled, x_valid_scaled, x_test_scaled


def error(y_true, y_pred, metric, task):
    if task == '회귀':
        if metric == 'mean_squared_error':
            return mean_squared_error(y_true, y_pred)
        elif metric == 'mean_absolute_error':
            return mean_absolute_error(y_true, y_pred)
        elif metric == 'root_mean_squared_error':
            return root_mean_squared_error(y_true, y_pred)
    else:
        if metric == 'log_loss (cross entropy)':
            return log_loss(y_true, y_pred)
        elif metric == 'accuracy':
            return accuracy_score(y_true, np.round(y_pred))


def plot_losses(train_losses, valid_losses, metric, epoch, n_epochs):
    plt.figure(figsize=(6, 4), facecolor='none')
    plt.plot(train_losses, label=f'Train Loss', color='cornflowerblue')
    plt.plot(valid_losses, label=f'Validation Loss', color='salmon')
    plt.title(f'Loss During Training (Epoch {epoch+1} / {n_epochs})')
    plt.xlabel('Epochs')
    plt.xticks(np.arange(0, n_epochs+1, step=(n_epochs//10)))
    plt.xlim(-n_epochs*0.05, n_epochs*1.05)
    plt.ylabel(f'Loss ({metric})')
    plt.grid(False)
    plt.legend()
    plt.show()


def plot_final_results(task, y_train, y_train_pred, y_valid, y_valid_pred, y_test, y_test_pred, all_classes=None):
    if task == '회귀':
        def plot_scatter(ax, y_true, y_pred, set_name):
            ax.scatter(y_true, y_pred, alpha=0.5, c = 'teal')
            ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2, c = 'firebrick')
            ax.set_title(f'True vs Predicted [{set_name} Set]')
            ax.set_xlabel('True Values')
            ax.set_ylabel('Predicted Values')
            r2 = r2_score(y_true, y_pred)

            mse = mean_squared_error(y_true, y_pred)
            rmse = root_mean_squared_error(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)

            return mse, rmse, mae, r2

        fig, axs = plt.subplots(1, 3, figsize=(12, 4))

        train_mse, train_rmse, train_mae, train_r2 = plot_scatter(axs[0], y_train, y_train_pred, 'Train')
        valid_mse, valid_rmse, valid_mae, valid_r2 = plot_scatter(axs[1], y_valid, y_valid_pred, 'Validation')
        test_mse, test_rmse, test_mae, test_r2 = plot_scatter(axs[2], y_test, y_test_pred, 'Test')

        print(f"""
  ✳️ Train Set
  -----------------------
  🔹 MSE  : {round(train_mse, 4)}
  🔹 RMSE : {round(train_rmse, 4)}
  🔹 MAE  : {round(train_mae, 4)}
  🔹 R2   : {round(train_r2, 4)}

  ❇️ Valid Set
  -----------------------
  🔹 MSE  : {round(valid_mse, 4)}
  🔹 RMSE : {round(valid_rmse, 4)}
  🔹 MAE  : {round(valid_mae, 4)}
  🔹 R2   : {round(valid_r2, 4)}

  ✴️ Test Set
  -----------------------
  🔸 MSE  : {round(test_mse, 4)}
  🔸 RMSE : {round(test_rmse, 4)}
  🔸 MAE  : {round(test_mae, 4)}
  🔸 R2   : {round(test_r2, 4)}
     """)

        plt.tight_layout()
        plt.show()

    else:
        def plot_conf_matrix(ax, y_true, y_pred, set_name):
            cm = confusion_matrix(y_true, y_pred, labels=all_classes, normalize='true')

            sns.heatmap(cm, annot=True, fmt='.2f', cmap=custom_cmap, xticklabels=all_classes, yticklabels=all_classes, ax=ax)
            ax.set_title(f'Confusion Matrix [{set_name} Set]')
            ax.set_xlabel('Predicted Label')
            ax.set_ylabel('True Label')

            accuracy = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
            recall = recall_score(y_true, y_pred, average='weighted')
            f1 = f1_score(y_true, y_pred, average='weighted')

            return accuracy, precision, recall, f1

        fig, axs = plt.subplots(1, 3, figsize=(12, 4))

        train_accuracy, train_precision, train_recall, train_f1 = plot_conf_matrix(axs[0], y_train, y_train_pred, 'Train')
        valid_accuracy, valid_precision, valid_recall, valid_f1 = plot_conf_matrix(axs[1], y_valid, y_valid_pred, 'Validation')
        test_accuracy, test_precision, test_recall, test_f1 = plot_conf_matrix(axs[2], y_test, y_test_pred, 'Test')

        print(f"""
  ✳️ Train Set
  -----------------------
  🔹 Accuracy  : {round(train_accuracy, 4)}
  🔹 Precision : {round(train_precision, 4)}
  🔹 Recall    : {round(train_recall, 4)}
  🔹 F1-score  : {round(train_f1, 4)}

  ❇️ Valid Set
  -----------------------
  🔹 Accuracy  : {round(valid_accuracy, 4)}
  🔹 Precision : {round(valid_precision, 4)}
  🔹 Recall    : {round(valid_recall, 4)}
  🔹 F1-score  : {round(valid_f1, 4)}

  ✴️ Test Set
  -----------------------
  🔸 Accuracy  : {round(test_accuracy, 4)}
  🔸 Precision : {round(test_precision, 4)}
  🔸 Recall    : {round(test_recall, 4)}
  🔸 F1-score  : {round(test_f1, 4)}
     """)

        plt.tight_layout()
        plt.show()

def start_training(b):
    global model, scaler

    model = create_model(model_info)
    n_epochs = model_info['epochs']
    lr = model_info['learning_rate']

    x_train_scaled, x_valid_scaled, x_test_scaled = preprocess_data(model_info, x_train, x_valid, x_test, scaler)

    first_fit = True

    train_losses = []
    valid_losses = []
    metric = model_info['metric']
    task = model_info['task']

    all_classes = np.unique(y_train)

    if model_info['model_type'] == '랜덤 포레스트':
        model.fit(x_train_scaled, y_train)

        clear_output(wait=True)

        y_train_pred = model.predict(x_train_scaled)
        y_valid_pred = model.predict(x_valid_scaled)
        y_test_pred = model.predict(x_test_scaled)
    else:
        for epoch in range(model_info['epochs']):
            if model_info['task'] == '회귀':
                model.partial_fit(x_train_scaled, y_train)
                y_train_pred = model.predict(x_train_scaled)
                y_valid_pred = model.predict(x_valid_scaled)
                train_loss = error(y_train, y_train_pred, metric, task)
                valid_loss = error(y_valid, y_valid_pred, metric, task)
            else:
                if first_fit:
                    model.partial_fit(x_train_scaled, y_train, classes=all_classes)
                    first_fit = False
                else:
                    model.partial_fit(x_train_scaled, y_train)

                y_train_pred = model.predict(x_train_scaled)
                y_valid_pred = model.predict(x_valid_scaled)

                if hasattr(model, "predict_proba"):
                    try:
                      train_loss = error(y_train, model.predict_proba(x_train_scaled), metric, task)
                      valid_loss = error(y_valid, model.predict_proba(x_valid_scaled), metric, task)
                    except:
                      print('학습률이 너무 높아 학습 과정에서 문제가 발생했습니다! 학습률을 낮춰보세요.')
                else:
                    train_loss = accuracy_score(y_train, y_train_pred)
                    valid_loss = accuracy_score(y_valid, y_valid_pred)

            train_losses.append(train_loss)
            valid_losses.append(valid_loss)

            # print every 10 epochs
            if epoch % 10 == 0:
                clear_output(wait=True)
                plot_losses(train_losses, valid_losses, metric, epoch, n_epochs)

        clear_output(wait=True)
        plot_losses(train_losses, valid_losses, metric, epoch, n_epochs)

        y_test_pred = model.predict(x_test_scaled)

        if model_info['task'] == '회귀':
            test_loss = error(y_test, y_test_pred, metric, task)
        else:
            if hasattr(model, "predict_proba"):
                test_loss = log_loss(y_test, model.predict_proba(x_test_scaled), labels=all_classes)
            else:
                test_loss = accuracy_score(y_test, y_test_pred)

    # 최종 손실 출력 (Train & Validation)
    print(f"""
  🎉 학습이 모두 완료되었습니다!
    """)
    time.sleep(1)
    print(f"""  🙌 이제 학습 결과를 확인합니다.
    """)
    time.sleep(1)

    print_info(result=True)

    if model_info['model_type'] == '랜덤 포레스트':
        print(f"""
  📢 최종 학습 결과
  -----------------------------""")
    else:
        print(f"""
  📢 최종 학습 결과
  -----------------------------
  📉 Metric : {metric}
  🔸 Final Train Loss : {train_losses[-1]:.4f}
  🔸 Final Valid Loss : {valid_losses[-1]:.4f}
  🏅 Final  Test Loss : {test_loss:.4f}""")

    plot_final_results(task, y_train, y_train_pred, y_valid, y_valid_pred, y_test, y_test_pred, all_classes)

train_possible = print_info()

if train_possible:
    start_button = widgets.Button(
        description="✨ Start Training",
        button_style = 'success',
        layout=widgets.Layout(width='300px', margin='10px 0px 0px 0px')
        )
    start_button.style.button_color = '#00AA98'


    start_button.on_click(start_training)
    display(start_button)

-----------------------------------------------------------------------------------------------